## **Práctica 1**
### **Parte 1.**
---
#### Encargado: **Marcos Bermejo Agenjo**
---


Consiste en la programación de un script que resuelva un problema de programación lineal. Se puede introducir ya en forma matricial sin necesidad de ir capturando una por una las entradas, como si de un software comercial se tratase.

Se trata de programar el algoritmo **«Símplex completo»** visto en clase. Decidirá automáticamente si empezar con Símplex primal o dual y cambiará si fuera necesario. **No hace falta considerar casos degenerados.**

---
##### **Solución:**

#### Entrada del algoritmo

Se considera que el programa recibe un problema de programación lineal en forma matricial, **sin incluir inicialmente las variables de holgura o superávit**.

Por ejemplo, dado el problema:

$$
\max z = 5x_1 + 4x_2
$$

Sujeto a:

$$
\begin{cases}
4x_1 + 2x_2 \leq 8 \\
x_1 + x_2 \leq 3 \\
x_1,x_2 \geq 0
\end{cases}
$$

La entrada se representa mediante:

$$
c = \begin{pmatrix} 5 & 4 \end{pmatrix},
\qquad
A = \begin{pmatrix}
4 & 2 \\
1 & 1
\end{pmatrix},
\qquad
b = \begin{pmatrix} 8 \\ 3 \end{pmatrix}
$$

Además, se especifican el **sentido de optimización** (maximización o minimización) y el **tipo de cada restricción** ($\leq$ o $\geq$).

El algoritmo se encargará de introducir las variables auxiliares necesarias y construir la matriz del Símplex.



#### Estructura de la implementación

Para implementar el algoritmo **Símplex completo**, dividiremos el programa en cuatro funciones principales y una función auxiliar.

##### 1. `preprocesar()`

Recibe el problema de programación lineal en su formulación original:

$$
\begin{aligned}
\operatorname{opt}\quad & z = c^T x \\
\text{sujeto a}\quad & Ax \;(\leq,\geq)\; b \\
& x \geq 0
\end{aligned}
$$

Donde $\operatorname{opt}$ indica maximización o minimización.

Su objetivo es preparar el problema para aplicar el algoritmo:

- Identificar el sentido de optimización.
- Examinar los signos y tipos de las restricciones.
- Introducir las variables de holgura o superávit necesarias.
- Construir la matriz inicial del Símplex.
- Identificar la base inicial y las transformaciones realizadas.

##### 2. `simplex_estandar()`

Implementa el algoritmo Símplex estándar o primal.

Partiendo de una **solución básica factible**, realiza iterativamente:

1. Comprobación de la condición de optimalidad.
2. Selección de la variable de entrada.
3. Selección de la variable de salida.
4. Actualización de la base.
5. Repetición hasta alcanzar una condición de parada.

##### 3. `simplex_dual()`

Implementa el algoritmo Símplex dual.

Partiendo de una **tabla dual factible**, realiza iterativamente:

1. Comprobación de la factibilidad primal.
2. Selección de la variable de salida.
3. Selección de la variable de entrada.
4. Actualización de la base.
5. Repetición hasta alcanzar una condición de parada.

##### 4. `simplex_completo()`

Función principal encargada de coordinar el procedimiento.

1. Llama a `preprocesar()`.
2. Comprueba las condiciones de la matriz inicial.
3. Decide si comenzar mediante Símplex estándar o dual.
4. Ejecuta el método correspondiente.
5. Cambia de método cuando sea necesario y se cumplan sus condiciones de aplicación.
6. Devuelve la solución en términos del problema original.

**Nota:** El procedimiento combinado estudiado en clase no garantiza la resolución de todas las formulaciones posibles, especialmente si la tabla inicial no es primal ni dual factible.

##### 5. `actualizar_matriz()` (auxiliar)

Ambos métodos comparten la operación de actualización de la matriz a partir de la base actual.

Siguiendo la formulación de los apuntes, cada iteración se calcula utilizando la **matriz original**, en lugar de la matriz obtenida en la iteración anterior.

Sea $B$ la matriz formada por las columnas básicas de $A$, y $c_B$ el vector de coeficientes de las variables básicas.

La matriz correspondiente a la base actual se obtiene mediante:

$$
\boxed{
\begin{pmatrix}
1 & c_B^T B^{-1} \\
0 & B^{-1}
\end{pmatrix}
\begin{pmatrix}
1 & -c^T & 0 \\
0 & A & b
\end{pmatrix}
}
$$

Por tanto, esta función será compartida por `simplex_estandar()` y `simplex_dual()`, evitando duplicar cálculos.


#### Esquema general

$$
\boxed{\text{Problema original}}
$$

$$
\downarrow
$$

$$
\boxed{\texttt{preprocesar()}}
$$

$$
\downarrow
$$

$$
\boxed{\texttt{simplex\_completo()}}
$$

$$
\downarrow
$$

$$
\begin{cases}
\texttt{simplex\_estandar()} & \text{si hay factibilidad primal}\\
\texttt{simplex\_dual()} & \text{si hay factibilidad dual}
\end{cases}
$$

Ambos métodos utilizan `actualizar_matriz()`.

El resultado final incluirá la solución, el valor de la función objetivo y el estado de terminación del algoritmo.


In [1]:
import numpy as np

In [2]:
def actualizar_matriz(M0: np.ndarray, base: list):
    n_rest = M0.shape[0] - 1
    
    # Extraemos la matriz básica B usando los índices de la base en M0
    B = M0[1:, base]
    
    assert B.shape == (n_rest, n_rest)
    
    # Inversa de B
    B_inv = np.linalg.inv(B)
    
    # Coeficientes de las variables básicas en la función objetivo
    c_B = -M0[0, base]
    
    # Construimos la matriz de transformación T
    fila_T = np.hstack(([1], c_B @ B_inv))
    resto_T = np.hstack((np.zeros((n_rest, 1)), B_inv))
    T = np.vstack((fila_T, resto_T))
    
    # Calculamos la nueva matriz M
    M = T @ M0
    
    return M

In [3]:
def preprocesar(opt: str, c: np.ndarray, A: np.ndarray, rest: list[str], b: np.ndarray):
    
    # Comprobamos que la entrada es válida
    if A.ndim != 2 or b.ndim != 1 or c.ndim != 1:
        raise ValueError("Dimensiones de la entrada incorrectas.")
    if opt not in ("max", "min"):
        raise ValueError("La optimización debe ser 'max' o 'min'.")
    if any(r not in ("<=", ">=") for r in rest):
        raise ValueError("Las restricciones deben ser '<=' o '>='.")
    if len(rest) != len(b) or len(b) != len(A):
        raise ValueError("El número de restricciones no coincide.")
    if len(c) != A.shape[1]:
        raise ValueError("El número de variables no coincide.")
    
    
    n_rest = len(rest)
    n_var = len(c)
    c_, A_, b_ = c.copy(), A.copy(), b.copy()
        
    # Convertimos todas las restricciones a '<='
    for i in range(n_rest):
        if rest[i] == ">=":
            A_[i] *= -1
            b_[i] *= -1
            
    # Añadimos variables de holgura
    I_n = np.eye(n_rest)
    A_ = np.hstack((A_, I_n))
    c_ = np.hstack((c_, np.zeros(n_rest)))
    
    # Calculamos la matriz inicial
    renglon_z = np.hstack(([1], -c_, [0]))
    restricciones = np.hstack((np.zeros((n_rest, 1)),
                               A_,
                               b_.reshape(-1, 1)
                               ))
    M0 = np.vstack((renglon_z, restricciones))
    
    return opt, M0

In [4]:
def simplex_estandar(M0: np.ndarray, base: list = None, max_iter: int = 1000):
    estado = ''
    nf, nc = M0.shape
    n_rest = nf - 1
    n_var = nc - n_rest - 2
    
    # Si no definimos la base (iteración 0)
    if base is None:
    # Calculamos los índices (comenzamos en x=(0,...,0))
        base_ = [i for i in range(n_var+1, n_rest + n_var + 1)]
    else:
        base_ = base.copy()
    
    # Actualizamos la matriz
    M = actualizar_matriz(M0, base_)
    
    it = 0
    while it < max_iter:
        no_base = [i for i in range(1,nc-1) if i not in base_]
        
        # Calculamos la variable de entrada
        mas_neg = 0
        ent = -1
        for i in no_base:
            actual = M[0, i]
            if actual < mas_neg:
                mas_neg = actual
                ent = i
                
        # Criterio de parada: no quedan coeficientes negativos en el renglón z
        if mas_neg == 0:
            estado = 'optimo'
            return M, base_, estado

        # Calculamos variable de salida
        menor = float('inf')
        for j in range(1, nf):
            if M[j, ent] <= 0:
                continue
            cociente = M[j, -1] / M[j, ent]
            if cociente < menor:
                menor = cociente
                sal = j
        
        if menor == float('inf'):
            estado = 'no acotado'
            return M, base_, estado
        
        # Modificamos la posición correspondiente a la fila de la matriz de entrada,
        # ajustando el índice
        base_[sal-1] = ent
        
        # Utilizamos la matriz original con la base modificada
        M = actualizar_matriz(M0, base_)
        
        it += 1
        
    estado = 'max_iter'        
    return M, base_, estado

In [5]:
matriz, base, estado = simplex_estandar(np.array([
                            [1, -5, -4, 0, 0, 0], 
                            [0, 4, 2, 1, 0, 8], 
                            [0, 1, 1, 0, 1, 3]]))
print(matriz, base, estado)

[[ 1.   0.   0.   0.5  3.  13. ]
 [ 0.   1.   0.   0.5 -1.   1. ]
 [ 0.   0.   1.  -0.5  2.   2. ]] [1, 2] optimo


In [6]:
def simplex_dual(M0: np.ndarray, base: list = None, max_iter: int = 1000):
    estado = ''
    nf, nc = M0.shape
    n_rest = nf - 1
    n_var = nc - n_rest - 2
    
    # Si no definimos la base (iteración 0)
    if base is None:
    # Calculamos los índices (comenzamos en x=(0,...,0))
        base_ = [i for i in range(n_var+1, n_rest + n_var + 1)]
    else:
        base_ = base.copy()
    
    # Actualizamos la matriz
    M = actualizar_matriz(M0, base_)
    
    it = 0
    while it < max_iter:
        no_base = [i for i in range(1,nc-1) if i not in base_]
        
        # Calculamos la variable de salida
        mas_neg = 0
        sal = -1
        for i in range(1, nf):
            actual = M[i, -1]
            if actual < mas_neg:
                mas_neg = actual
                sal = i
                
        # Criterio de parada: no quedan términos independientes negativos
        if mas_neg == 0:
            estado = 'optimo'
            return M, base_, estado

        # Calculamos variable de entrada
        menor = float('inf')
        for j in no_base:
            if M[sal, j] >= 0:
                continue
            cociente = np.abs(M[0, j] / M[sal, j])
            if cociente < menor:
                menor = cociente
                ent = j
        
        if menor == float('inf'):
            estado = 'infactible'
            return M, base_, estado
        
        # Modificamos la posición correspondiente a la fila de la matriz de entrada,
        # ajustando el índice
        base_[sal-1] = ent
        
        # Utilizamos la matriz original con la base modificada
        M = actualizar_matriz(M0, base_)
        
        it += 1
        
    estado = 'max_iter'        
    return M, base_, estado

In [8]:
matriz, base, estado = simplex_dual(np.array([
                            [1, -8, -3, 0, 0, 0], 
                            [0, -4, -1, 1, 0, -5], 
                            [0, -2, -1, 0, 1, -4]]))
print(matriz, base, estado)

[[ 1.   0.   0.  -1.  -2.  13. ]
 [ 0.   1.   0.  -0.5  0.5  0.5]
 [ 0.   0.   1.   1.  -2.   3. ]] [1, 2] optimo


In [ ]:
def simplex_completo(opt, c, A, rest, b):